In [ ]:
# Importar las librerías necesarias para conectar con SQL Server y procesar datos.
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [ ]:
# Cargar la configuración y crear las conexiones a las bases de datos Stage y DW.
# Cargar variables de entorno desde el archivo .env
load_dotenv()

# Stage
cadena_stage = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS')};"
    f"UID={os.getenv('SQLSERVER_USUARIO')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD')};"
    f"TrustServerCertificate=yes;"
)

engine_stage = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_stage}"
)


# DW

cadena_dw = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER_DW')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR_DW')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS_DW')};"
    f"UID={os.getenv('SQLSERVER_USUARIO_DW')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD_DW')};"
    f"TrustServerCertificate=yes;"
)

engine_dw = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_dw}"
)

In [ ]:
# Extraer del Stage las fechas de venta distintas para construir la dimensión.
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")

consulta_fecha = f"""
SELECT DISTINCT
    CAST(fecha_venta AS DATE) AS fecha
FROM {esquema_stage}.Venta
WHERE fecha_venta IS NOT NULL
"""

df_fecha = pd.read_sql(
    consulta_fecha,
    engine_stage
)

print("Fechas extraídas:", len(df_fecha))

df_fecha.head()

Fechas extraídas: 1336


,fecha
0,2024-05-03
1,2024-04-10
2,2024-09-03
3,2024-05-26
4,2024-09-26


In [ ]:
# Convertir la columna de fechas al tipo datetime para derivar sus atributos.
df_fecha["fecha"] = pd.to_datetime(
    df_fecha["fecha"]
)

In [ ]:
# Generar la clave sustituta de cada fecha con formato YYYYMMDD.
df_fecha["fecha_sk"] = (
    df_fecha["fecha"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [ ]:
# Obtener el día, mes, trimestre y año de cada fecha.
df_fecha["dia"] = df_fecha["fecha"].dt.day

df_fecha["mes"] = df_fecha["fecha"].dt.month

df_fecha["trimestre"] = df_fecha["fecha"].dt.quarter

df_fecha["anio"] = df_fecha["fecha"].dt.year

In [ ]:
# Asignar el nombre en español correspondiente a cada número de mes.
meses = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}

df_fecha["nombre_mes"] = (
    df_fecha["mes"].map(meses)
)

In [ ]:
# Marcar si cada fecha corresponde a sábado o domingo.
df_fecha["es_fin_semana"] = (
    df_fecha["fecha"].dt.dayofweek >= 5
)

In [ ]:
# Seleccionar y ordenar las columnas que conforman la dimensión fecha.
df_fecha = df_fecha[
    [
        "fecha_sk",
        "fecha",
        "dia",
        "mes",
        "nombre_mes",
        "trimestre",
        "anio",
        "es_fin_semana"
    ]
].copy()

In [ ]:
# Convertir los valores datetime a objetos date para la carga al DW.
df_fecha["fecha"] = df_fecha["fecha"].dt.date

In [ ]:
# Mostrar una vista previa de los registros preparados para la dimensión.
df_fecha.head()

,fecha_sk,fecha,dia,mes,nombre_mes,trimestre,anio,es_fin_semana
0,20240503,2024-05-03,3,5,Mayo,2,2024,False
1,20240410,2024-04-10,10,4,Abril,2,2024,False
2,20240903,2024-09-03,3,9,Septiembre,3,2024,False
3,20240526,2024-05-26,26,5,Mayo,2,2024,True
4,20240926,2024-09-26,26,9,Septiembre,3,2024,False


In [ ]:
# Consultar las claves de fecha que ya están registradas en la dimensión del DW.
esquema_dw = os.getenv("SQLSERVER_ESQUEMA_DW")

df_fechas_existentes = pd.read_sql(
    f"""
    SELECT fecha_sk
    FROM {esquema_dw}.DimFecha
    """,
    engine_dw
)

print(
    "Fechas existentes en el DW:",
    len(df_fechas_existentes)
)

Fechas existentes en el DW: 0


In [ ]:
# Filtrar las fechas cuya clave todavía no existe en la dimensión del DW.
df_fechas_nuevas = df_fecha[
    ~df_fecha["fecha_sk"].isin(
        df_fechas_existentes["fecha_sk"]
    )
].copy()

print(
    "Fechas nuevas:",
    len(df_fechas_nuevas)
)

Fechas nuevas: 1336


In [ ]:
# Insertar en la dimensión las fechas nuevas, si existen.
if not df_fechas_nuevas.empty:

    df_fechas_nuevas.to_sql(
        name="DimFecha",
        con=engine_dw,
        schema=esquema_dw,
        if_exists="append",
        index=False
    )

    print(
        f"{len(df_fechas_nuevas)} fechas nuevas cargadas."
    )

else:
    print("No existen fechas nuevas para cargar.")

1336 fechas nuevas cargadas.


In [ ]:
# Consultar y mostrar los primeros 20 registros de la dimensión cargada.
df_resultado = pd.read_sql(
    f"""
    SELECT TOP 20 *
    FROM {esquema_dw}.DimFecha
    ORDER BY fecha_sk
    """,
    engine_dw
)

display(df_resultado)

,fecha_sk,fecha,dia,mes,nombre_mes,trimestre,anio,es_fin_semana
0,20230101,2023-01-01,1,1,Enero,1,2023,True
1,20230102,2023-01-02,2,1,Enero,1,2023,False
2,20230103,2023-01-03,3,1,Enero,1,2023,False
3,20230104,2023-01-04,4,1,Enero,1,2023,False
4,20230105,2023-01-05,5,1,Enero,1,2023,False
5,20230106,2023-01-06,6,1,Enero,1,2023,False
6,20230107,2023-01-07,7,1,Enero,1,2023,True
7,20230108,2023-01-08,8,1,Enero,1,2023,True
8,20230109,2023-01-09,9,1,Enero,1,2023,False
9,20230110,2023-01-10,10,1,Enero,1,2023,False
